In [1]:
from datasets import load_dataset

dataset = load_dataset("openbmb/UltraFeedback", split="train")
print(dataset.column_names)

['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers']


In [2]:
print(dataset)
print(dataset.column_names)

Dataset({
    features: ['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers'],
    num_rows: 63967
})
['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers']


In [3]:
import json
print(json.dumps(dataset[0], indent=2))

{
  "source": "evol_instruct",
  "instruction": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}",
  "models": [
    "alpaca-7b",
    "pythia-12b",
    "starchat",
    "vicuna-33b"
  ],
  "completions": [
    {
      "annotations": {
        "helpfulness": {
          "Rating": "2",
          "Rationale": "The response is clear and not lengthy, but it lacks useful and comprehensive information.",
          "Rationale For Rating": "The code is partially incorrect as it checks if the country name ends with \"Mediterranean\" instead of checking if it borders the Mediterranean Sea, whic

In [4]:
print(json.dumps(dataset[0]["completions"][0],indent=2))
print(type(dataset[0]["completions"][0]))
print(dataset[0]["completions"][0]["response"])

{
  "annotations": {
    "helpfulness": {
      "Rating": "2",
      "Rationale": "The response is clear and not lengthy, but it lacks useful and comprehensive information.",
      "Rationale For Rating": "The code is partially incorrect as it checks if the country name ends with \"Mediterranean\" instead of checking if it borders the Mediterranean Sea, which may cause confusion.",
      "Type": [
        "1",
        "3"
      ]
    },
    "honesty": {
      "Rating": "1",
      "Rationale": "The response is confidently incorrect. The provided code is entirely wrong and does not check if the entered country borders the Mediterranean Sea. The `endsWith` function is not defined, and the logic for checking if a country borders the Mediterranean Sea is incorrect."
    },
    "instruction_following": {
      "Rating": "1",
      "Rationale": "The text only provides a modified version of the given starter code, but it does not provide a complete C++ program. The code does not check if the e

In [9]:
def extract_example(instruction:str,completion:dict):
    try:
        keys=["instruction","response","helpfulness","truthfulness","instruction_following","overall"]
        result=dict.fromkeys(keys,None)
        result["instruction"]=instruction
        result["response"]=completion["response"]
        for key in keys[2:5]:
            keys1=["score","rationale"]
            dic=dict.fromkeys(keys1,None)
            dic["score"]=int(completion["annotations"][key]["Rating"])
            dic["rationale"]=completion["annotations"][key]["Rationale"]
            result[key]=dic
        dic=dict.fromkeys(["score","rationale"],None)
        dic["score"]=round(completion["overall_score"]/2)
        dic["rationale"]=completion["critique"]
        result["overall"]=dic

        return result
    except (KeyError,TypeError,ValueError):
        return None

In [10]:
result = extract_example(
    dataset[0]["instruction"],
    dataset[0]["completions"][0]
)
print(json.dumps(result, indent=2))

{
  "instruction": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}",
  "response": "int main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    if (endsWith(country, \"Mediterranean\")) {\n        cout << \"Yes, the country \" << country\n             << \" borders the Mediterranean Sea.\";\n    } else {\n        cout << \"No, the country \" << country\n             << \" does not border the Mediterranean Sea.\";\n    }\n    return 0;\n}",
  "helpfuln

In [11]:
def process_dataset(dataset):
    examples = []
    skipped = 0

    for row in dataset:
        instruction = row["instruction"]
        for completion in row["completions"]:
            result = extract_example(instruction, completion)
            if result is None:
                skipped += 1
            else:
                examples.append(result)

    print(f"Total extracted: {len(examples)}")
    print(f"Total skipped: {skipped}")
    return examples

In [12]:
all_examples = process_dataset(dataset)

Total extracted: 255862
Total skipped: 2


In [13]:
import random

random.seed(42)
sampled = random.sample(all_examples, 9000)

train_examples = sampled[:8000]
holdout_examples = sampled[8000:]

print(f"Train: {len(train_examples)}")
print(f"Holdout: {len(holdout_examples)}")

Train: 8000
Holdout: 1000


In [14]:
import json

with open("train.json", "w") as f:
    json.dump(train_examples, f, indent=2)

with open("holdout.json", "w") as f:
    json.dump(holdout_examples, f, indent=2)

print("Saved.")

Saved.


In [15]:
import json

SYSTEM_PROMPT = """You are LitmusAI, an expert judge that evaluates the quality of AI-generated responses.

Given an instruction and a response, evaluate the response on three criteria:
- Helpfulness (1-5): Does the response genuinely help the user? Does it address their need completely?
- Truthfulness (1-5): Are all factual claims accurate? Does the response avoid hallucinations?
- Instruction Following (1-5): Does the response do exactly what the instruction asked, in the format requested?
- Overall (1-5): A holistic judgment of the response quality across all dimensions.

Score 1 = very poor, 2 = poor, 3 = acceptable, 4 = good, 5 = excellent.

Respond ONLY in valid JSON with exactly this schema:
{"helpfulness":{"score":<int>,"rationale":"<str>"},"truthfulness":{"score":<int>,"rationale":"<str>"},"instruction_following":{"score":<int>,"rationale":"<str>"},"overall":{"score":<int>,"rationale":"<str>"}}

No preamble. No explanation. JSON only."""


def format_example(example: dict) -> str:
    scorecard = {
        "helpfulness": {
            "score": example["helpfulness"]["score"],
            "rationale": example["helpfulness"]["rationale"]
        },
        "truthfulness": {
            "score": example["truthfulness"]["score"],
            "rationale": example["truthfulness"]["rationale"]
        },
        "instruction_following": {
            "score": example["instruction_following"]["score"],
            "rationale": example["instruction_following"]["rationale"]
        },
        "overall": {
            "score": example["overall"]["score"],
            "rationale": example["overall"]["rationale"]
        }
    }

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>

Instruction: {example['instruction']}
Response: {example['response']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{json.dumps(scorecard)}<|eot_id|>"""

    return prompt


def format_dataset(examples: list) -> list:
    return [{"text": format_example(ex)} for ex in examples]

In [16]:
train_formatted = format_dataset(train_examples)
holdout_formatted = format_dataset(holdout_examples)

# Verify one example looks right
print(train_formatted[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are LitmusAI, an expert judge that evaluates the quality of AI-generated responses.

Given an instruction and a response, evaluate the response on three criteria:
- Helpfulness (1-5): Does the response genuinely help the user? Does it address their need completely?
- Truthfulness (1-5): Are all factual claims accurate? Does the response avoid hallucinations?
- Instruction Following (1-5): Does the response do exactly what the instruction asked, in the format requested?
- Overall (1-5): A holistic judgment of the response quality across all dimensions.

Score 1 = very poor, 2 = poor, 3 = acceptable, 4 = good, 5 = excellent.

Respond ONLY in valid JSON with exactly this schema:
{"helpfulness":{"score":<int>,"rationale":"<str>"},"truthfulness":{"score":<int>,"rationale":"<str>"},"instruction_following":{"score":<int>,"rationale":"<str>"},"overall":{"score":<int>,"rationale":"<str>"}}

No preamble. No explanation. JSON only.<

In [17]:
with open("train_formatted.json", "w") as f:
    json.dump(train_formatted, f, indent=2)

with open("holdout_formatted.json", "w") as f:
    json.dump(holdout_formatted, f, indent=2)

print("Saved.")

Saved.
